<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/03-mathematical-foundations.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)


## **Mathematical Foundations for Machine Learning**

Machine learning combines geometry, probability, differentiation, information, and numerical computation. These are not separate prerequisites that disappear once modeling begins. They describe five different aspects of the same system:

- **linear algebra** describes data, parameters, transformations, and geometric structure;
- **probability and statistics** describe uncertainty, sampling, estimation, and predictions;
- **calculus** describes sensitivity and how parameters should change;
- **information theory** measures uncertainty and disagreement between distributions;
- **numerical computation** determines whether an algebraically correct method can be computed reliably and efficiently.

This chapter is an ML-oriented bridge rather than a replacement for the dedicated [Linear Algebra](../Math/Linear-Algebra.html), [Single-Variable Calculus](../Math/Single-Variable-Calculus.html), and [Statistics](../Statistics/Statistics.html) notes. The goal is to make model equations readable. For every object, ask: What is its shape? What does it represent? Which assumptions make the operation valid? What changes geometrically or probabilistically? Where will this appear again in a learning algorithm?

The chapter deliberately uses small numerical experiments. A formula becomes much easier to trust when its shape can be checked, its geometric condition can be measured, and an analytic gradient can be compared with a finite difference.


### **Notation and Shapes**

Mathematical notation compresses a large amount of structural information. In machine learning, a shape error is often a conceptual error: samples and features have been swapped, a batch dimension has disappeared, or a scalar loss has been confused with a vector of per-example losses.

#### **Scalars, Vectors, Matrices, and Tensors**

A **scalar** is one number, such as a learning rate $\eta$, a loss $L$, or a regularization strength $\lambda$. A **vector** is an ordered collection, such as one feature vector $x\in\mathbb{R}^{d}$ or parameter vector $w\in\mathbb{R}^{d}$. A **matrix** has two axes, such as a data matrix $X\in\mathbb{R}^{n\times d}$. A **tensor** generalizes the idea to more axes: an image batch may have shape `(batch, height, width, channels)`, while a sequence batch may have shape `(batch, time, embedding)`.

Subscripts select components. $x_i$ is the $i$-th entry of a vector; $X_{ij}$ is row $i$, column $j$ of a matrix. Superscripts should be interpreted from context: $x^{(i)}$ often means sample $i$, while $X^\top$ means transpose. A transpose exchanges axes, so if $X\in\mathbb{R}^{n\times d}$, then $X^\top\in\mathbb{R}^{d\times n}$.

![Scalars, vectors, matrices, and tensors carry different shape information; matrix multiplication keeps the unmatched outer dimensions.](assets/ml-matrix-shapes.svg){fig-alt="Diagram of scalar, vector, matrix, tensor, and shape-checked matrix-vector multiplication"}

Shape reasoning makes matrix multiplication almost mechanical. If

$$
A\in\mathbb{R}^{m\times n}, \qquad B\in\mathbb{R}^{n\times p},
$$

then the inner dimensions $n$ match and $AB\in\mathbb{R}^{m\times p}$. The entry $(AB)_{ij}$ is the inner product between row $i$ of $A$ and column $j$ of $B$. Element-wise multiplication is different: it combines aligned entries and normally requires identical or broadcast-compatible shapes.

Broadcasting is a programming convention that virtually repeats a smaller array across a larger one. Adding a bias $b\in\mathbb{R}^{k}$ to a batch of scores $S\in\mathbb{R}^{n\times k}$ adds the same $k$-dimensional bias to every row. Broadcasting is useful, but an unintended broadcast can silently produce a plausible-looking wrong result.

#### **Datasets and Design Matrices**

For supervised learning,

$$
\mathcal{D}=\{(x^{(i)},y^{(i)})\}_{i=1}^{n}
$$

denotes a dataset of $n$ examples. Stacking row vectors gives the design matrix

$$
X=
\begin{bmatrix}
(x^{(1)})^\top\\
\vdots\\
(x^{(n)})^\top
\end{bmatrix}
\in\mathbb{R}^{n\times d}.
$$

Here $n$ is the sample count and $d$ is the feature count. For scalar regression, $y\in\mathbb{R}^{n}$. For $k$-class one-hot labels, $Y\in\{0,1\}^{n\times k}$. A linear predictor computes (Xw+b), where $w\in\mathbb{R}^{d}$, $Xw\in\mathbb{R}^{n}$, and scalar $b$ is broadcast across samples.

Some texts store examples as columns instead. Neither convention is inherently superior, but every derivation and codebase must be internally consistent. This blog uses samples as rows unless stated otherwise.


<details>
<summary><strong>Python example: inspect shapes and verify a batched linear model</strong></summary>

```python
import numpy as np

# Four samples, each described by three features.
X = np.array(
    [
        [1.0, 0.2, 3.0],
        [1.0, 1.1, 2.0],
        [1.0, 2.4, 0.5],
        [1.0, 3.2, 1.3],
    ]
)
w = np.array([0.5, -1.0, 0.8])
b = 0.25

# (4 x 3) @ (3,) -> (4,), then the scalar bias is broadcast.
scores = X @ w + b

assert X.shape[1] == w.shape[0]
assert scores.shape == (X.shape[0],)

print("X shape:", X.shape)
print("w shape:", w.shape)
print("scores shape:", scores.shape)
print("scores:", scores.round(3))
```

</details>


### **Linear Algebra for Models and Data**

Linear algebra provides both a representation language and a geometric language. A matrix can be read as stored numbers, as a collection of feature directions, as a function that transforms vectors, or as a dataset whose correlations reveal lower-dimensional structure. Different interpretations become useful in different models.

#### **Vector Spaces, Bases, and Projections**

A **vector space** is a collection closed under vector addition and scalar multiplication. The **span** of vectors $v_1,\ldots,v_k$ is every linear combination

$$
\operatorname{span}(v_1,\ldots,v_k)
=\left\{\sum_{j=1}^{k}a_jv_j:a_j\in\mathbb{R}\right\}.
$$

A **basis** is a linearly independent set that spans the space. It supplies coordinates: the same geometric vector has different numerical coordinates under different bases. Features act like coordinate directions, and learned representations can be understood as changing basis so that important variation becomes easier to separate or compress.

An orthogonal projection finds the closest point in a subspace. If $Q$ has orthonormal columns, the projection of $b$ onto $\operatorname{Col}(Q)$ is $QQ^\top b$. For a full-column-rank matrix $A$, the projection matrix is

$$
P=A(A^\top A)^{-1}A^\top.
$$

Least squares chooses $\hat{x}$ so that $p=A\hat{x}$ is the closest prediction to $b$:

$$
\hat{x}=\arg\min_x\|b-Ax\|_2^2.
$$

At the optimum, the residual $e=b-A\hat{x}$ is orthogonal to every column of $A$, giving the normal equations $A^\top A\hat{x}=A^\top b$. Geometrically, there is no remaining residual direction that can be reduced by moving along a model feature.

![Least squares decomposes an observed target into a projection in the model's column space and an orthogonal residual.](assets/projection-least-squares.svg){fig-alt="Least squares projection with target vector, fitted projection, and perpendicular residual"}

*Concept adapted from [MIT OpenCourseWare, Projection Matrices and Least Squares](https://ocw.mit.edu/courses/18-06sc-linear-algebra-fall-2011/pages/least-squares-determinants-and-eigenvalues/projection-matrices-and-least-squares/).*


<details>
<summary><strong>Python example: verify the geometry of least squares</strong></summary>

```python
import numpy as np

# Fit a line b approximately equal to intercept + slope * t.
t = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
A = np.column_stack([np.ones_like(t), t])
b = np.array([1.1, 2.0, 2.9, 4.2, 4.8])

# lstsq uses a stable factorization rather than explicitly inverting A.T @ A.
x_hat, squared_residuals, rank, singular_values = np.linalg.lstsq(A, b, rcond=None)
projection = A @ x_hat
residual = b - projection

print("[intercept, slope]:", x_hat.round(4))
print("fitted values:", projection.round(4))
print("residual:", residual.round(4))
print("A.T @ residual:", (A.T @ residual).round(12))
print("rank:", rank, "singular values:", singular_values.round(4))
```

</details>

#### **Norms, Inner Products, and Distances**

An inner product measures alignment. In Euclidean space, $x^\top z=\sum_jx_jz_j$. It determines length through $|x|_2=\sqrt{x^\top x}$ and angle through

$$
\cos\theta=\frac{x^\top z}{\|x\|_2\|z\|_2}.
$$

A norm measures vector magnitude. The general $p$-norm is

$$
\|x\|_p=\left(\sum_{j=1}^{d}|x_j|^p\right)^{1/p}.
$$

Important cases are $|x|_1=\sum_j|x_j|$, Euclidean $|x|_2$, and $|x|_\infty=\max_j|x_j|$. Distances are often norms of differences, $d(x,z)=|x-z|$, as introduced in Chapter 02.

Norm choice also defines regularization geometry. An L2 penalty discourages large coefficients smoothly and tends to distribute weight among correlated features. An L1 penalty has corners in its constraint region, making exact zero coefficients more likely. Norms therefore express what counts as a "small" parameter vector, not merely how length is calculated.


<details>
<summary><strong>Python example: compare norms, alignment, and scale</strong></summary>

```python
import numpy as np

x = np.array([3.0, -4.0, 0.5])
z = np.array([1.0, -1.0, 2.0])

l1 = np.linalg.norm(x, ord=1)
l2 = np.linalg.norm(x, ord=2)
linf = np.linalg.norm(x, ord=np.inf)
cosine = (x @ z) / (np.linalg.norm(x) * np.linalg.norm(z))

print("L1 norm:", round(l1, 3))
print("L2 norm:", round(l2, 3))
print("L-infinity norm:", round(linf, 3))
print("dot product:", round(x @ z, 3))
print("cosine similarity:", round(cosine, 3))
```

</details>

#### **Rank, Inverses, and Pseudoinverses**

The **rank** of a matrix is the number of linearly independent rows or columns. For a design matrix, low rank means some feature direction can be constructed from others. Parameters may then be non-identifiable: several coefficient vectors produce the same predictions.

A square matrix $A\in\mathbb{R}^{d\times d}$ has an inverse only when it has full rank. The inverse satisfies $A^{-1}A=AA^{-1}=I$. In learning code, explicitly constructing an inverse is usually less stable and less efficient than solving a linear system with `solve`, `lstsq`, QR, or SVD.

The Moore-Penrose pseudoinverse $A^+$ extends inversion to rectangular or rank-deficient matrices. It gives a least-squares solution $x=A^+b$; when many solutions fit equally well, it returns the minimum-L2-norm solution. Pseudoinverses do not make missing information reappear. Small singular values still make estimates highly sensitive to noise.


<details>
<summary><strong>Python example: solve a rank-deficient system with a pseudoinverse</strong></summary>

```python
import numpy as np

# Column 2 is exactly twice column 1, so the matrix has rank 1 rather than 2.
A = np.array([[1.0, 2.0], [2.0, 4.0], [3.0, 6.0]])
b = np.array([1.0, 2.0, 3.0])

print("rank:", np.linalg.matrix_rank(A))

try:
    normal_equation_solution = np.linalg.solve(A.T @ A, A.T @ b)
except np.linalg.LinAlgError as error:
    print("normal equations fail:", error)

minimum_norm_solution = np.linalg.pinv(A) @ b
print("pseudoinverse solution:", minimum_norm_solution.round(4))
print("prediction:", (A @ minimum_norm_solution).round(4))
print("residual norm:", round(np.linalg.norm(b - A @ minimum_norm_solution), 12))
```

</details>

#### **Eigenvalues and Eigendecomposition**

For a square matrix $A$, a non-zero vector $v$ is an eigenvector when

$$
Av=\lambda v.
$$

The transformation changes $v$'s magnitude by eigenvalue $\lambda$ without changing its direction, apart from a possible sign reversal. A real symmetric matrix has orthonormal eigenvectors and real eigenvalues, so it can be decomposed as $A=Q\Lambda Q^\top$.

Covariance matrices and Hessians are symmetric. For a covariance matrix, eigenvectors identify orthogonal directions of variation and eigenvalues measure variance along those directions. For a Hessian, eigenvalues measure local curvature: large positive values indicate steep directions, small values indicate flat directions, and negative values indicate directions of local descent from a saddle point.

A matrix is positive semidefinite when $x^\top A x\geq0$ for every $x$, equivalent to non-negative eigenvalues for a symmetric matrix. Covariance matrices are positive semidefinite because variance cannot be negative.


<details>
<summary><strong>Python example: recover principal variance directions from a covariance matrix</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(5)
base = rng.normal(size=500)
X = np.column_stack(
    [
        2.0 * base + rng.normal(scale=0.3, size=500),
        0.8 * base + rng.normal(scale=0.2, size=500),
    ]
)
X_centered = X - X.mean(axis=0)
covariance = np.cov(X_centered, rowvar=False)

# eigh is specialized for symmetric matrices and returns ascending eigenvalues.
eigenvalues, eigenvectors = np.linalg.eigh(covariance)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

first_direction = eigenvectors[:, 0]
projected_variance = np.var(X_centered @ first_direction, ddof=1)

print("covariance:\n", covariance.round(3))
print("eigenvalues:", eigenvalues.round(3))
print("first eigenvector:", first_direction.round(3))
print("variance along it:", round(projected_variance, 3))
```

</details>

#### **Singular Value Decomposition**

Eigendecomposition applies to square matrices and is especially well behaved for symmetric ones. Singular value decomposition applies to any $A\in\mathbb{R}^{m\times n}$:

$$
A=U\Sigma V^\top.
$$

In a reduced SVD of rank $r$, $U\in\mathbb{R}^{m\times r}$ contains orthonormal output directions, $V\in\mathbb{R}^{n\times r}$ contains orthonormal input directions, and diagonal $\Sigma\in\mathbb{R}^{r\times r}$ contains singular values $\sigma_1\geq\cdots\geq\sigma_r>0$. The operation $V^\top$ changes to singular-vector coordinates, $\Sigma$ scales each direction, and $U$ places the result in output space.

![SVD interprets a matrix as a rotation or reflection, an axis-aligned stretch, and a final rotation or reflection.](assets/svd-rotate-stretch.svg){fig-alt="Four-stage geometric interpretation of SVD from circle to rotated ellipse"}

*Concept adapted from [MIT OpenCourseWare, Singular Value Decomposition](https://ocw.mit.edu/courses/18-065-matrix-methods-in-data-analysis-signal-processing-and-machine-learning-spring-2018/resources/lecture-6-singular-value-decomposition-svd/).*

Keeping the largest $k$ singular values gives

$$
A_k=\sum_{j=1}^{k}\sigma_j u_jv_j^\top,
$$

the best rank-$k$ approximation under the Frobenius and spectral norms. This underlies PCA, latent semantic analysis, matrix completion, denoising, compression, and low-rank parameterization. The pseudoinverse follows by replacing each non-zero $\sigma_j$ with $1/\sigma_j$; tiny singular values therefore amplify noise.

| Decomposition | Matrix requirement | Main ML interpretation |
|---|---|---|
| Eigen $A=Q\Lambda Q^{-1}$ | square and diagonalizable; simplest for symmetric $A$ | invariant directions, covariance variance, Hessian curvature |
| SVD $A=U\Sigma V^\top$ | any rectangular matrix | input/output directions, rank, compression, stable least squares |


<details>
<summary><strong>Python example: measure low-rank SVD reconstruction error</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(9)

# Construct a matrix with two strong latent factors plus small noise.
left_factors = rng.normal(size=(40, 2))
right_factors = rng.normal(size=(2, 20))
A = left_factors @ right_factors + 0.08 * rng.normal(size=(40, 20))

U, singular_values, Vt = np.linalg.svd(A, full_matrices=False)
original_norm = np.linalg.norm(A, ord="fro")

for rank_k in [1, 2, 5, 10]:
    approximation = (
        U[:, :rank_k]
        @ np.diag(singular_values[:rank_k])
        @ Vt[:rank_k, :]
    )
    relative_error = np.linalg.norm(A - approximation, ord="fro") / original_norm
    print(f"rank {rank_k:2d} relative error: {relative_error:.4f}")

print("leading singular values:", singular_values[:6].round(3))
```

</details>


### **Probability and Statistics**

Probability specifies a model of uncertain outcomes; statistics reasons from observed samples back to the process or parameters that generated them. In ML, a classifier may output a conditional distribution, a loss may be a negative log-likelihood, and an evaluation estimate is itself uncertain because it is computed from finite data.

#### **Random Variables and Common Distributions**

A random variable $X$ maps outcomes to numerical values. A discrete variable has a probability mass function $p(x)=P(X=x)$. A continuous variable has a density $p(x)$, where probabilities are integrals over intervals; the density at one point is not itself a probability.

| Distribution | Support and parameters | Typical ML role |
|---|---|---|
| Bernoulli | $x\in\{0,1\}$, success probability $p$ | binary labels, binary likelihood |
| Categorical | one of $K$ classes, probability vector $\pi$ | multiclass prediction |
| Binomial | success count in $n$ Bernoulli trials | aggregate event counts |
| Gaussian | $x\in\mathbb{R}$, mean $\mu$, variance $\sigma^2$ | residual model, latent variables, approximations |
| Poisson | non-negative count, rate $\lambda$ | event counts per exposure interval |
| Exponential | positive waiting time, rate $\lambda$ | time between memoryless events |
| Beta | $p\in(0,1)$, shape $\alpha,\beta$ | prior or uncertainty over a probability |

A distribution is an assumption about support, shape, and variability. Choosing a Gaussian likelihood for a positive, strongly skewed count is not harmless notation; it defines which errors the model considers plausible.


<details>
<summary><strong>Python example: compare empirical moments with distribution parameters</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(11)
sample_size = 100_000

bernoulli = rng.binomial(n=1, p=0.3, size=sample_size)
poisson = rng.poisson(lam=4.0, size=sample_size)
gaussian = rng.normal(loc=2.0, scale=1.5, size=sample_size)

cases = {
    "Bernoulli(p=0.3)": (bernoulli, 0.3, 0.3 * 0.7),
    "Poisson(lambda=4)": (poisson, 4.0, 4.0),
    "Gaussian(mu=2, sd=1.5)": (gaussian, 2.0, 1.5 ** 2),
}

for name, (sample, theoretical_mean, theoretical_variance) in cases.items():
    print(
        name,
        {
            "sample_mean": round(sample.mean(), 3),
            "theoretical_mean": theoretical_mean,
            "sample_variance": round(sample.var(), 3),
            "theoretical_variance": theoretical_variance,
        },
    )
```

</details>

#### **Joint, Marginal, and Conditional Probability**

A joint distribution $p(x,y)$ describes two variables together. A marginal removes a variable by summing or integrating:

$$
p(x)=\sum_y p(x,y).
$$

A conditional distribution restricts attention to cases where $Y=y$:

$$
p(x\mid y)=\frac{p(x,y)}{p(y)}, \qquad p(y)>0.
$$

The product rule rearranges this definition:

$$
p(x,y)=p(x\mid y)p(y)=p(y\mid x)p(x).
$$

Marginalization combines competing explanations; conditioning updates the relevant sample space. In supervised learning, the empirical dataset approximates a joint process $p(x,y)$, while discriminative models directly learn $p(y\mid x)$ or a decision derived from it.


<details>
<summary><strong>Python example: derive marginals and conditionals from a joint table</strong></summary>

```python
import numpy as np

# Rows: weather {dry, rain}; columns: transport {walk, bus}.
joint = np.array(
    [
        [0.42, 0.18],
        [0.08, 0.32],
    ]
)

p_weather = joint.sum(axis=1)
p_transport = joint.sum(axis=0)
p_bus_given_rain = joint[1, 1] / p_weather[1]
p_rain_given_bus = joint[1, 1] / p_transport[1]

print("P(weather):", p_weather)
print("P(transport):", p_transport)
print("P(bus | rain):", round(p_bus_given_rain, 3))
print("P(rain | bus):", round(p_rain_given_bus, 3))
print("Conditionals are different because their denominators differ.")
```

</details>

#### **Expectation, Variance, and Covariance**

Expectation is a probability-weighted average. For a discrete variable,

$$
\mathbb{E}[X]=\sum_x x\,p(x).
$$

It describes a distribution, not necessarily a value that can occur. Expectation is linear even when variables are dependent: $\mathbb{E}[aX+bY]=a\mathbb{E}[X]+b\mathbb{E}[Y]$.

Variance measures squared deviation from the mean:

$$
\operatorname{Var}(X)=\mathbb{E}[(X-\mathbb{E}[X])^2].
$$

Covariance measures whether two variables move together:

$$
\operatorname{Cov}(X,Y)=\mathbb{E}[(X-\mu_X)(Y-\mu_Y)].
$$

Positive covariance indicates aligned deviations, negative covariance opposing deviations, and zero covariance no linear association. Zero covariance does not generally imply independence. Correlation normalizes covariance by standard deviations and is unitless, but it still captures only a particular form of association.

For a random vector $X\in\mathbb{R}^d$, covariance matrix $\Sigma\in\mathbb{R}^{d\times d}$ stores all pairwise covariances. Its diagonal contains variances, and it determines the elliptical geometry of a multivariate Gaussian.


<details>
<summary><strong>Python example: distinguish covariance, correlation, and independence</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(13)
x = rng.normal(size=20_000)
y_linear = 2.0 * x + rng.normal(scale=0.5, size=x.size)
y_nonlinear = x ** 2

linear_covariance = np.cov(x, y_linear, ddof=1)[0, 1]
linear_correlation = np.corrcoef(x, y_linear)[0, 1]
nonlinear_correlation = np.corrcoef(x, y_nonlinear)[0, 1]

print("Cov(x, 2x + noise):", round(linear_covariance, 3))
print("Corr(x, 2x + noise):", round(linear_correlation, 3))
print("Corr(x, x^2):", round(nonlinear_correlation, 3))
print("x and x^2 are dependent even though their linear correlation is near zero.")
```

</details>

#### **Bayes' Rule and Conditional Independence**

Bayes' rule reverses a conditional probability:

$$
p(\theta\mid D)=\frac{p(D\mid\theta)p(\theta)}{p(D)}.
$$

The prior $p(\theta)$ describes beliefs or frequencies before observing data $D$. The likelihood $p(D\mid\theta)$ scores how compatible the data are with a candidate parameter. The evidence $p(D)=\int p(D\mid\theta)p(\theta)d\theta$ normalizes all candidate explanations. The posterior $p(\theta\mid D)$ is the updated distribution.

Likelihood is not a probability distribution over $\theta$ unless normalized. This distinction matters: $p(\text{positive}\mid\text{disease})$ is test sensitivity, while $p(\text{disease}\mid\text{positive})$ is the clinically relevant posterior and depends strongly on prevalence.

![A rare-event example shows how prior prevalence and both positive-test pathways determine the posterior.](assets/bayes-update.svg){fig-alt="Bayesian update from prior population through true and false positive likelihoods to posterior"}

*Concept based on [MIT OpenCourseWare, Conditional Probability, Independence, and Bayes' Theorem](https://ocw.mit.edu/courses/18-05-introduction-to-probability-and-statistics-spring-2022/resources/mit18_05_s22_lec03_pdf/).*

Variables $X$ and $Y$ are conditionally independent given $Z$ when

$$
p(x,y\mid z)=p(x\mid z)p(y\mid z).
$$

Conditional independence is weaker and more useful than unconditional independence. Naive Bayes assumes features are conditionally independent given the class; graphical models encode many such assumptions to factorize a large joint distribution.


<details>
<summary><strong>Python example: compute a rare-event posterior with Bayes' rule</strong></summary>

```python
prior = 0.01
sensitivity = 0.90
false_positive_rate = 0.05

# Total probability of a positive result under both possible states.
p_positive = (
    sensitivity * prior
    + false_positive_rate * (1.0 - prior)
)
posterior = sensitivity * prior / p_positive

print("P(positive):", round(p_positive, 4))
print("P(condition | positive):", round(posterior, 4))
print("Posterior percentage:", f"{100 * posterior:.1f}%")
```

</details>

#### **Sampling and Estimation**

A population quantity such as $\mu$ is a parameter; a statistic such as sample mean $\bar{X}$ is a random variable because it changes across samples. An **estimator** is a rule that maps data to an estimate. Bias asks whether its expected estimate equals the target. Variance asks how much estimates change across samples. Consistency asks whether the estimate approaches the target as sample size grows.

Maximum likelihood estimation chooses parameters that maximize $p(D\mid\theta)$, usually by minimizing negative log-likelihood. Maximum a posteriori estimation maximizes $p(D\mid\theta)p(\theta)$, connecting priors to regularization. These point estimates hide uncertainty unless accompanied by intervals, posterior distributions, bootstrap variation, or repeated-split analysis.

The law of large numbers explains why sample averages stabilize. The central limit theorem explains why many standardized sample means become approximately Gaussian under suitable conditions, even when the original observations are skewed. It does not say that the raw data become Gaussian, nor does it rescue dependent or heavy-tailed sampling without checking assumptions.


<details>
<summary><strong>Python example: observe the sampling distribution of a skewed mean</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(17)
repetitions = 20_000

def skewness(values):
    centered = values - values.mean()
    return np.mean(centered ** 3) / np.mean(centered ** 2) ** 1.5

for sample_size in [1, 5, 30, 100]:
    # Each row is one repeated sample from an exponential population.
    samples = rng.exponential(scale=1.0, size=(repetitions, sample_size))
    sample_means = samples.mean(axis=1)
    print(
        f"n={sample_size:3d}",
        "mean=", round(sample_means.mean(), 3),
        "std=", round(sample_means.std(), 3),
        "skew=", round(skewness(sample_means), 3),
    )
```

</details>


### **Calculus and Matrix Calculus**

Training asks how a scalar objective changes when thousands or billions of parameters change. Calculus supplies local sensitivity; matrix calculus organizes those sensitivities without expanding every coordinate.

#### **Partial Derivatives and Gradients**

For $f:\mathbb{R}^d\rightarrow\mathbb{R}$, the partial derivative $\partial f/\partial x_j$ changes coordinate $j$ while holding the others fixed. The gradient stacks all partial derivatives:

$$
\nabla f(x)=
\begin{bmatrix}
\frac{\partial f}{\partial x_1}&\cdots&\frac{\partial f}{\partial x_d}
\end{bmatrix}^{\!\top}.
$$

The directional derivative along unit vector $u$ is $u^\top\nabla f(x)$. The gradient points toward the steepest local increase under Euclidean distance, so $-\nabla f$ is the steepest local descent direction.

For mean squared error

$$
L(w)=\frac{1}{n}\|Xw-y\|_2^2,
$$

the residual $Xw-y\in\mathbb{R}^n$, and

$$
\nabla_wL=\frac{2}{n}X^\top(Xw-y)\in\mathbb{R}^d.
$$

The shapes tell the story: $X^\top\in\mathbb{R}^{d\times n}$ aggregates per-example residual signals back into one gradient per feature weight.


<details>
<summary><strong>Python example: check a matrix-calculus gradient with finite differences</strong></summary>

```python
import numpy as np

X = np.array([[1.0, 0.2], [1.0, 1.3], [1.0, 2.1], [1.0, 3.0]])
y = np.array([1.0, 2.2, 2.8, 4.1])
w = np.array([0.4, 0.9])

def mse(parameters):
    residual = X @ parameters - y
    return np.mean(residual ** 2)

analytic_gradient = (2.0 / len(y)) * X.T @ (X @ w - y)

epsilon = 1e-6
numeric_gradient = np.zeros_like(w)
for index in range(len(w)):
    step = np.zeros_like(w)
    step[index] = epsilon
    numeric_gradient[index] = (mse(w + step) - mse(w - step)) / (2 * epsilon)

print("analytic gradient:", analytic_gradient.round(8))
print("numeric gradient: ", numeric_gradient.round(8))
print("maximum difference:", np.max(np.abs(analytic_gradient - numeric_gradient)))
```

</details>

#### **Jacobians and Hessians**

When $f:\mathbb{R}^d\rightarrow\mathbb{R}^m$ has a vector output, its Jacobian $J\in\mathbb{R}^{m\times d}$ stores every output-input derivative:

$$
J_{ij}=\frac{\partial f_i}{\partial x_j}.
$$

Each row describes how one output changes; each column describes how all outputs respond to one input. Deep-learning libraries usually avoid materializing full Jacobians. They compute products such as vector-Jacobian products during reverse-mode automatic differentiation.

For scalar $f$, the Hessian is the Jacobian of the gradient:

$$
H_{ij}=\frac{\partial^2f}{\partial x_i\partial x_j}.
$$

It describes local curvature and parameter interactions. Newton's method uses $H^{-1}\nabla f$ to scale the update by curvature, but full Hessians are expensive and may be indefinite. Quasi-Newton and Hessian-vector methods approximate the useful curvature information.


<details>
<summary><strong>Python example: inspect a Jacobian and Hessian by shape</strong></summary>

```python
import numpy as np

x1, x2 = 2.0, -1.0

# f(x1, x2) = [x1*x2, x1^2 + sin(x2)] has a 2 x 2 Jacobian.
jacobian = np.array(
    [
        [x2, x1],
        [2.0 * x1, np.cos(x2)],
    ]
)

# g(x) = 0.5*x^T H x + c^T x has constant Hessian H.
hessian = np.array([[4.0, 1.0], [1.0, 2.0]])
curvatures = np.linalg.eigvalsh(hessian)

print("Jacobian shape:", jacobian.shape)
print(jacobian.round(3))
print("\nHessian:\n", hessian)
print("Hessian eigenvalues:", curvatures.round(3))
print("Positive eigenvalues mean this quadratic is strictly convex.")
```

</details>

#### **The Chain Rule**

Models are compositions. If $q=g(x)$ and $L=f(q)$, the scalar chain rule is

$$
\frac{dL}{dx}=\frac{dL}{dq}\frac{dq}{dx}.
$$

The first factor is the upstream sensitivity of the final loss to the intermediate value; the second is the local sensitivity of that intermediate value to its input. Backpropagation applies this rule from output to input while reusing cached forward values.

![A forward computation evaluates values from inputs to output; the backward pass propagates gradients in reverse through local derivatives.](assets/chain-rule-computation.svg){fig-alt="Stanford CS231n computation graph for f equals x plus y times z with forward values and backward gradients"}

*Source: [Stanford CS231n, Backpropagation and the Chain Rule](https://cs231n.github.io/optimization-2/).*

For vector functions, Jacobians compose. Reverse-mode automatic differentiation is efficient when many parameters feed one scalar loss because it propagates a vector-Jacobian product without constructing each full Jacobian. At a branch, gradient contributions add because changing the shared variable affects the loss through every outgoing path.


<details>
<summary><strong>Python example: perform a scalar forward and backward pass manually</strong></summary>

```python
# Function: f = (x + y) * z
x, y, z = -2.0, 5.0, -4.0

# Forward pass caches the intermediate q.
q = x + y
f = q * z

# Backward pass starts with df/df = 1 and applies local derivatives.
df = 1.0
dq = z * df       # df/dq = z
dz = q * df       # df/dz = q
dx = 1.0 * dq     # dq/dx = 1
dy = 1.0 * dq     # dq/dy = 1

print("forward: q =", q, "f =", f)
print("gradients: df/dx =", dx, "df/dy =", dy, "df/dz =", dz)
```

</details>

#### **Taylor Approximation**

Taylor expansion approximates a smooth function near $x$. For a small displacement $\Delta$,

$$
f(x+\Delta)\approx f(x)+\nabla f(x)^\top\Delta
+\frac{1}{2}\Delta^\top H(x)\Delta.
$$

The first-order term predicts change from slope; the second-order term corrects for curvature. Gradient descent relies on a local linear model plus a sufficiently small step. Newton methods use the quadratic model. A large update can leave the region where either approximation is accurate, explaining why learning rate and trust-region control matter.


<details>
<summary><strong>Python example: compare first- and second-order Taylor approximations</strong></summary>

```python
import numpy as np

# Approximate exp(x) around x0 = 0, where f = f' = f'' = 1.
def first_order(delta):
    return 1.0 + delta

def second_order(delta):
    return 1.0 + delta + 0.5 * delta ** 2

for delta in [0.05, 0.5, 1.0]:
    exact = np.exp(delta)
    linear_error = abs(exact - first_order(delta))
    quadratic_error = abs(exact - second_order(delta))
    print(
        f"delta={delta:.2f}",
        f"exact={exact:.5f}",
        f"first_error={linear_error:.5f}",
        f"second_error={quadratic_error:.5f}",
    )
```

</details>


### **Information Theory**

Information theory treats uncertainty and distributional mismatch quantitatively. In ML it appears in decision-tree splitting, classification losses, variational inference, representation learning, language modeling, and feature selection.

#### **Entropy and Cross-Entropy**

For a discrete distribution $p=(p_1,\ldots,p_K)$, entropy is

$$
H(p)=-\sum_{k=1}^{K}p_k\log p_k.
$$

Each $p_k$ is an outcome probability. The convention $0\log0=0$ follows by continuity. Natural logarithms measure entropy in nats; base-2 logarithms measure bits. Entropy is low when one outcome is nearly certain and maximal for a uniform distribution.

![Pure labels have zero entropy, mixed labels have intermediate entropy, and a balanced binary set has maximum entropy.](assets/entropy-levels.png){fig-alt="Google diagram comparing zero, low, and high entropy label groups"}

*Source: [Google for Developers, Exact Splitter for Binary Classification](https://developers.google.com/machine-learning/decision-forests/binary-classification).*

Cross-entropy evaluates probabilities $q$ when outcomes follow $p$:

$$
H(p,q)=-\sum_{k=1}^{K}p_k\log q_k.
$$

For a one-hot target with true class $c$, all terms vanish except $-\log q_c$. Assigning low probability to the observed class creates a large loss; assigning zero probability gives infinite loss. Cross-entropy is therefore sensitive to confidence, not only whether the top class is correct.


<details>
<summary><strong>Python example: inspect binary entropy and one-hot cross-entropy</strong></summary>

```python
import numpy as np

def binary_entropy(p):
    probabilities = np.array([p, 1.0 - p], dtype=float)
    positive = probabilities > 0
    return -np.sum(probabilities[positive] * np.log2(probabilities[positive]))

for p in [0.0, 0.1, 0.25, 0.5, 0.9, 1.0]:
    print(f"p={p:.2f}, entropy={binary_entropy(p):.3f} bits")

true_class = 2
predicted = np.array([0.05, 0.15, 0.70, 0.10])
cross_entropy = -np.log(predicted[true_class])
print("\nOne-hot cross-entropy:", round(cross_entropy, 4), "nats")
```

</details>

#### **KL Divergence**

Kullback-Leibler divergence measures the extra log-loss from representing $p$ with $q$:

$$
D_{\mathrm{KL}}(p\|q)=\sum_k p_k\log\frac{p_k}{q_k}.
$$

It satisfies $D_{\mathrm{KL}}(p\|q)\geq0$ and equals zero only when the distributions agree almost everywhere. It is not a distance: generally $D_{\mathrm{KL}}(p\|q)\neq D_{\mathrm{KL}}(q\|p)$, and it does not obey the triangle inequality. If $p_k>0$ where $q_k=0$, the divergence is infinite.

Cross-entropy decomposes as

$$
H(p,q)=H(p)+D_{\mathrm{KL}}(p\|q).
$$

When the target distribution $p$ is fixed, minimizing cross-entropy over $q$ also minimizes KL divergence. The asymmetry matters in variational inference: different KL directions penalize missing modes and covering low-density regions differently.


<details>
<summary><strong>Python example: verify cross-entropy equals entropy plus KL divergence</strong></summary>

```python
import numpy as np

p = np.array([0.60, 0.30, 0.10])
q = np.array([0.45, 0.45, 0.10])

entropy_p = -np.sum(p * np.log(p))
cross_entropy_pq = -np.sum(p * np.log(q))
kl_pq = np.sum(p * np.log(p / q))
kl_qp = np.sum(q * np.log(q / p))

print("H(p):", round(entropy_p, 6))
print("H(p, q):", round(cross_entropy_pq, 6))
print("KL(p || q):", round(kl_pq, 6))
print("H(p) + KL(p || q):", round(entropy_p + kl_pq, 6))
print("KL(q || p):", round(kl_qp, 6), "(different direction)")
```

</details>

#### **Mutual Information**

Mutual information measures how far a joint distribution is from independence:

$$
I(X;Y)=D_{\mathrm{KL}}\big(p(x,y)\|p(x)p(y)\big)
=\sum_{x,y}p(x,y)\log\frac{p(x,y)}{p(x)p(y)}.
$$

It is zero exactly when $X$ and $Y$ are independent and is symmetric despite being built from KL divergence. Equivalent identities include $I(X;Y)=H(X)-H(X\mid Y)$: observing $Y$ reduces uncertainty about $X$ by this amount on average.

Mutual information can detect nonlinear dependence and supports filter-based feature selection, decision-tree information gain, and representation-learning objectives. Estimating it from finite continuous data is difficult; binning, density estimation, or neural estimators introduce bias and tuning choices.


<details>
<summary><strong>Python example: compare mutual information for independent and associated variables</strong></summary>

```python
import numpy as np

def mutual_information(joint):
    joint = np.asarray(joint, dtype=float)
    joint = joint / joint.sum()
    p_x = joint.sum(axis=1, keepdims=True)
    p_y = joint.sum(axis=0, keepdims=True)
    independent_reference = p_x @ p_y
    mask = joint > 0
    return np.sum(joint[mask] * np.log2(joint[mask] / independent_reference[mask]))

independent = np.array([[0.25, 0.25], [0.25, 0.25]])
associated = np.array([[0.45, 0.05], [0.05, 0.45]])

print("Independent MI:", round(mutual_information(independent), 6), "bits")
print("Associated MI:", round(mutual_information(associated), 6), "bits")
```

</details>


### **Numerical Computation**

Real computers use finite precision. Algebraic equivalence does not guarantee equal numerical behaviour, and algorithmic complexity determines whether an otherwise valid method can be used at scale.

#### **Conditioning and Floating-Point Error**

Floating-point numbers approximate a finite subset of real numbers. Rounding error is usually tiny, but an ill-conditioned problem can amplify it. For a full-rank matrix, the 2-norm condition number is

$$
\kappa_2(A)=\frac{\sigma_{\max}(A)}{\sigma_{\min}(A)}.
$$

A condition number near one means all directions are scaled similarly. A large value means at least one direction is nearly collapsed, so small input perturbations can cause large solution changes. Conditioning is a property of the problem; numerical stability is a property of the algorithm used to solve it.

For least squares, forming $X^\top X$ squares the condition number, which is one reason QR or SVD-based solvers are preferred. Other common hazards include subtractive cancellation, division by tiny values, overflow in exponentials, underflow in probability products, and explicit matrix inversion.


<details>
<summary><strong>Python example: observe sensitivity in an ill-conditioned linear system</strong></summary>

```python
import numpy as np

size = 10
row = np.arange(size)[:, None]
column = np.arange(size)[None, :]
hilbert = 1.0 / (row + column + 1.0)

true_solution = np.ones(size)
b = hilbert @ true_solution

# Apply a tiny relative perturbation to the observed right-hand side.
perturbation = np.zeros_like(b)
perturbation[-1] = 1e-10
estimated = np.linalg.solve(hilbert, b + perturbation)

relative_input_change = np.linalg.norm(perturbation) / np.linalg.norm(b)
relative_solution_change = np.linalg.norm(estimated - true_solution) / np.linalg.norm(true_solution)

print("condition number:", f"{np.linalg.cond(hilbert):.3e}")
print("relative input change:", f"{relative_input_change:.3e}")
print("relative solution change:", f"{relative_solution_change:.3e}")
```

</details>

#### **Log-Sum-Exp and Stable Probabilities**

Probability models often multiply many small probabilities. Working in log space changes products into sums and prevents underflow. A second key identity is

$$
\log\sum_i e^{z_i}
=m+\log\sum_i e^{z_i-m}, \qquad m=\max_i z_i.
$$

Subtracting $m$ makes every exponent non-positive, so the largest exponential is one. The same shift stabilizes softmax because adding a shared constant to all logits cancels between numerator and denominator.

![Direct exponentiation of large logits overflows, while subtracting the maximum gives identical softmax probabilities with bounded intermediate values.](assets/stable-softmax.svg){fig-alt="Side-by-side unstable and stable softmax computation"}

*Numerical principle reflected in [SciPy's `logsumexp` documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.logsumexp.html).*

Clipping probabilities can prevent `log(0)` in ad hoc code, but it changes the objective. Stable formulations such as log-sum-exp, `log_softmax`, and library cross-entropy routines preserve the intended mathematics more faithfully.


<details>
<summary><strong>Python example: compare naive and stable softmax</strong></summary>

```python
import numpy as np

logits = np.array([1000.0, 1001.0, 1002.0])

with np.errstate(over="ignore", invalid="ignore"):
    raw_exponentials = np.exp(logits)
    naive_softmax = raw_exponentials / raw_exponentials.sum()

shifted = logits - logits.max()
stable_exponentials = np.exp(shifted)
stable_softmax = stable_exponentials / stable_exponentials.sum()
stable_logsumexp = logits.max() + np.log(stable_exponentials.sum())

print("naive exponentials:", raw_exponentials)
print("naive softmax:", naive_softmax)
print("stable softmax:", stable_softmax.round(6))
print("stable log-sum-exp:", round(stable_logsumexp, 6))
```

</details>

#### **Vectorization and Computational Complexity**

Vectorization expresses many scalar operations as array operations that optimized numerical libraries can execute in compiled loops, SIMD instructions, and parallel hardware. For a batch $X\in\mathbb{R}^{n\times d}$ and weights $W\in\mathbb{R}^{d\times k}$, computing (XW) costs on the order of $O(ndk)$. Big-O suppresses constants but reveals which dimensions dominate growth.

Vectorization does not remove computational cost. A fully materialized pairwise distance matrix for $n$ examples needs $O(n^2)$ memory, often becoming the true bottleneck. Batching, sparse matrices, low-rank structure, iterative solvers, and approximate nearest-neighbour methods trade exactness or passes over data for manageable memory and time.

The squared Euclidean distance identity

$$
\|x_i-z_j\|_2^2=\|x_i\|_2^2+\|z_j\|_2^2-2x_i^\top z_j
$$

turns nested coordinate loops into matrix multiplication, but the final $n\times m$ result still occupies memory.


<details>
<summary><strong>Python example: vectorize pairwise squared distances and verify the result</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(23)
X = rng.normal(size=(5, 3))
Z = rng.normal(size=(4, 3))

# Broadcasting plus matrix multiplication computes every pair at once.
x_squared = np.sum(X ** 2, axis=1, keepdims=True)
z_squared = np.sum(Z ** 2, axis=1, keepdims=True).T
distances_vectorized = x_squared + z_squared - 2.0 * X @ Z.T

# A small explicit loop is useful as a correctness oracle.
distances_loop = np.empty((len(X), len(Z)))
for i, x in enumerate(X):
    for j, z in enumerate(Z):
        distances_loop[i, j] = np.sum((x - z) ** 2)

print("distance matrix shape:", distances_vectorized.shape)
print("maximum difference:", np.max(np.abs(distances_vectorized - distances_loop)))
print(distances_vectorized.round(3))
```

</details>


### **Mathematical Prerequisite Map**

The purpose of this chapter is not to memorize every identity before continuing. It is to recognize which tool a model relies on and know where to deepen it when a derivation becomes important.

| Foundation | Minimum working understanding | Reappears in this ML series |
|---|---|---|
| Shapes and matrix multiplication | trace sample, feature, class, and parameter dimensions | every model and implementation |
| Projections and least squares | interpret fitted values as a closest point in a model subspace | linear regression, kernels, optimization |
| Norms and inner products | connect geometry to distance and regularization | KNN, SVM, linear models, sparse learning |
| Rank, eigenvalues, and SVD | diagnose identifiability, curvature, and low-rank structure | PCA, matrix factorization, numerical solvers |
| Conditional probability and Bayes | distinguish likelihood, prior, evidence, and posterior | Bayesian learning, graphical models, calibration |
| Expectation and sampling | distinguish population risk from finite-sample estimates | statistical learning theory and evaluation |
| Gradients, Jacobians, and Hessians | interpret first- and second-order sensitivity | objectives, optimization, neural networks |
| Entropy, cross-entropy, KL, and MI | compare uncertainty and probability distributions | trees, classification, variational and representation learning |
| Conditioning and stable computation | separate mathematical correctness from reliable implementation | optimization, probabilistic models, ML systems |

A useful readiness test is practical: given an expression such as

$$
L(W)=-\frac{1}{n}\sum_{i=1}^{n}\log\operatorname{softmax}(XW)_{i,y_i},
$$

you should be able to identify the shapes of $X$ and $W$, interpret softmax as a conditional distribution, recognize negative log-likelihood as cross-entropy for one-hot labels, explain why log-softmax needs numerical stabilization, and understand that the chain rule propagates a scalar loss gradient back to every element of $W$. Later chapters will derive and apply each layer of that expression in context.

For fuller mathematical development, continue with [Linear Algebra](../Math/Linear-Algebra.html), [Single-Variable Calculus](../Math/Single-Variable-Calculus.html), and the [Statistics guideline](../Statistics/Statistics.html). The next machine-learning chapter uses these tools to ask a different question: why should low training error imply anything about unseen data?

[Back to Machine Learning guideline](Machine Learning.html)
